SCM.275x - Advanced Supply Chain Systems Planning and Network Design
# **Transportation Problem**

### *Before starting, make sure to save a copy of this notebook to your Google Drive!*

## **Initialization**

In [1]:
# # Install necessary packages if they are not already installed

# !pip install gurobipy   # Gurobi optimization solver
# !pip install pandas     # Pandas for data analysis and manipulation
# !pip install folium     # Folium for creating interactive maps
# !pip install geopy      # Geopy for computing distances and working with geographic data


In [40]:
# Import all required packages

import pandas as pd                   # For data manipulation and analysis
#import gurobipy as grb                # Gurobi optimization library for solving mathematical models
import mypulp as grb                # Gurobi optimization library for solving mathematical models
import folium                         # For creating interactive maps
import folium.plugins as plugins      # Additional plugins for folium
from geopy.distance import geodesic   # For calculating geodesic distances between two points


## **Helper functions**

### **Plotting nodes on a map**

In [41]:
# Defining a function to plot nodes on a map using folium

def plot_nodes(map,                         # Folium map object to plot the nodes on
               nodes,                       # Dictionary of node objects where each node contains attributes like latitude and longitude
               icon,                        # Icon symbol to use for the markers on the map
               color,                       # Color of the marker icon
               background_color,            # Background color of the marker icon
               ):

    # Loop through each node in the dictionary
    for node in nodes.values():

        # Create a folium marker
        marker = folium.Marker(
            location=[node.lat, node.lon],              # Set the marker's location
            popup = (node.ID + "-" + node.name),        # Create a marker popup with the node ID and name
            icon=plugins.BeautifyIcon(                  # Create a marker's icon
                icon=icon,
                icon_shape="circle",
                text_color=color,                       # Set the marker's color
                border_color=color,
                background_color=background_color,
            )
        )

        # Add a folium marker to the map
        marker.add_to(map)


### **Computing geodesic distance**

In [42]:
# Defining a function for computing geodesic distances between two locations

def compute_geodesic_distance(origin,       # Origin node object
                              destination,  # Destination node
                              unit='km'):   # Unit ('km' or 'mi'), default value = 'km'

    # Extract coordinates (latitude and longitude) from origin and destination
    origin_coordinates = [origin.lat, origin.lon]
    destination_coordinates = [destination.lat, destination.lon]

    # Compute distance based on the specified unit ('km' or 'mi')
    if unit == 'km':
        distance = geodesic(origin_coordinates, destination_coordinates).km  # Compute distance in kilometers
    elif unit == 'mi':
        distance = geodesic(origin_coordinates, destination_coordinates).mi  # Compute distance in miles

    return distance  # Return the calculated distance


### **Plotting flows on a map**

In [43]:
# Defining a function to plot flows on a map using folium

def plot_flows(map,                   # Folium map object where flows will be plotted.
               vars,                  # Dictionary of decision variables from the optimization model
               nodes,                 # Dictionary of node objects
               max_width = 30,        # Maximum line width for the flows, default is 30
               color = 'grey',        # Color of the lines representing flows, default is grey
               opacity = 0.5):        # Opacity of the lines, default is 0.5

    # Find the maximum flow value to normalize line widths
    max_val = max([var.X for (node1_key, node2_key), var in vars.items()])

    # Iterate over flow decision variables (keys represent node pairs)
    for (node1_key, node2_key), var in vars.items():

        # Plot only positive flows
        if var.X > 0:

            # Get the coordinates of the nodes for plotting the line
            points = [[nodes[node1_key].lat, nodes[node1_key].lon],
                      [nodes[node2_key].lat, nodes[node2_key].lon]]


            # Add a PolyLine to the map to represent the flow between the nodes
            folium.PolyLine(points,
                            color=color,                                # Set the color of the line
                            weight=var.X / max_val * max_width,         # Normalize line width based on flow value
                            opacity=opacity,                            # Set line opacity
                            popup=var.X).add_to(map)                    # Show the flow value in a popup on the map


## **Data setup and preprocessing**

### **Nodes**

#### Reading input files

In [44]:
# File containing customer data
customer_data_file = 'https://raw.githubusercontent.com/scm275/problem_sets_scm275/main/transportation_problem/customers.csv'

# Loading customer data into a pandas DataFrame
customers_df = pd.read_csv(customer_data_file)

# Displaying the first few rows of the DataFrame to verify the data
customers_df.head()

,ID,name,lat,lon,demand
0,C01,Concord,43.206898,-71.537994,9000
1,C02,Trenton,40.220596,-74.769913,10000
2,C03,Austin,30.274670,-97.740349,10000
3,C04,Providence,41.830914,-71.414963,9000
4,C05,Montpelier,44.262436,-72.580536,7000


In [45]:
# File containing supplier data
supplier_data_file = 'https://raw.githubusercontent.com/scm275/problem_sets_scm275/main/transportation_problem/suppliers.csv'

# Loading supplier data into a pandas DataFrame
suppliers_df = pd.read_csv(supplier_data_file)

# Displaying the first few rows of the DataFrame to verify the data
suppliers_df.head()


,ID,name,lat,lon,supply
0,S01,Columbia,34.000343,-81.033211,40000
1,S02,Jefferson City,38.579201,-92.172935,30000
2,S03,Santa Fe,35.682240,-105.939728,20000
3,S04,Richmond,37.538857,-77.433640,32000
4,S05,Carson City,39.163914,-119.766121,20000


#### Definition of Classes

In [46]:
# Class representing a Customer object

class Customer():
    def __init__(self, ID, name, lat, lon, demand):
        self.ID = ID              # Customer's ID
        self.name = name          # Customer's name
        self.lat = lat            # Customer's latitude
        self.lon = lon            # Customer's longitude
        self.demand = demand      # Customer's demand


In [47]:
# Class representing a Supplier object

class Supplier():
    def __init__(self, ID, name, lat, lon, supply):
        self.ID = ID            # Supplier's ID
        self.name = name        # Supplier's name
        self.lat = lat          # Supplier's latitude
        self.lon = lon          # Supplier's longitude
        self.supply = supply    # Supplier's available supply


#### Creating node objects

In [48]:
# Initializing an empty dictionary to store node objects
nodes = dict()

In [49]:
# Creating a dictionary of customer objects
customers = dict()
for i, row in customers_df.iterrows():
    customers[row['ID']] = Customer(ID=row['ID'],           # Customer's ID
                                    name=row['name'],       # Customer's name
                                    lat=row['lat'],         # Customer's latitude
                                    lon=row['lon'],         # Customer's longitude
                                    demand=row['demand'])   # Customer's demand

# Merging the customers dictionary into the existing nodes dictionary
nodes = {**nodes, **customers}

In [50]:
# Creating a dictionary of supplier objects
suppliers = dict()
for i, row in suppliers_df.iterrows():
    suppliers[row['ID']] = Supplier(ID=row['ID'],           # Supplier's ID
                                    name=row['name'],       # Supplier's name
                                    lat=row['lat'],         # Supplier's latitude
                                    lon=row['lon'],         # Supplier's longitude
                                    supply=row['supply'])   # Supplier's available supply

# Merging the suppliers dictionary into the existing nodes dictionary
nodes = {**customers, **suppliers}

In [51]:
# Verifying that the total supply is less than or equal to the total demand

# Initialize a variable to store the total supply from all suppliers
total_supply = 0
# Iterate over all supplier objects and sum up their supply values
for x in suppliers.values():
    total_supply += x.supply

# Initialize a variable to store the total demand from all customers
total_demand = 0
# Iterate over all customer objects and sum up their demand values
for x in customers.values():
    total_demand += x.demand

# Print the total supply value calculated
print('Total Supply:', total_supply)

# Print the total demand value calculated
print('Total Demand:', total_demand)


Total Supply: 171000
Total Demand: 171000


#### Visualizing node objects

In [92]:
# Create a new map centered on the US with a zoom level of 5
map = folium.Map([40, -95.0], zoom_start=5)

# Plot customer locations with a store icon, green color, and yellow background
plot_nodes(map=map, nodes=customers, icon='store', color='green', background_color='yellow')

# Plot supplier locations with an industry icon, orange color, and yellow background
plot_nodes(map=map, nodes=suppliers, icon='industry', color='orange', background_color='yellow')

# Add a tile layer for better map visualization (cartodbpositron theme)
folium.TileLayer('cartodbpositron').add_to(map)

# Display the map with all the plotted data
map


## pydeck入門

### 点の描画

**コードの説明**

*   **データの準備:**
    *   表示する点の経度と緯度を `pandas` の `DataFrame` で準備します。
*   **`ScatterplotLayer` の設定:**
    *   `pdk.Layer` を使用して、点の散布図を描画するためのレイヤーを作成します。
    *   `"ScatterplotLayer"` でレイヤーの種類を指定します。
    *   `data`:  描画するデータを指定します。
    *   `get_position`: 各点の位置を、データフレームのカラム名で指定します。
    *   `get_radius`: 点の半径をピクセル単位で指定します。
    *   `get_color`: 点の色をRGBで指定します。
    *   `pickable`: マウスオーバーやクリックなどのインタラクションを有効にします。
*   **`ViewState` の設定:**
    *   地図の初期表示位置とズームレベルを指定します。
    *   `latitude`（緯度）、`longitude`（経度）、`zoom`（ズーム）、`pitch`（傾き）を設定します。
*   **`Deck` オブジェクトの作成:**
    *   `pdk.Deck` で、地図全体のオブジェクトを作成します。
    *   `layers`: 描画するレイヤーをリストで指定します。
    *   `initial_view_state`: 初期表示設定を指定します。
    *   `tooltip`: マウスオーバー時に表示する情報を指定します。
*   **HTMLファイルへの保存:**
    *   作成した地図を `to_html` メソッドでHTMLファイルとして保存します。

### その他のレイヤーの例

PyDeckでは、点の描画以外にも、様々な種類のレイヤーをサポートしています。

*   **`LineLayer`**: 線を描画します。
*   **`PolygonLayer`**: ポリゴンを描画します。
*   **`GeoJsonLayer`**: GeoJSON形式のデータを描画します。
*   **`PointCloudLayer`**: 点群データを描画します。
*   **`TileLayer`**: タイルベースの地図を表示します。
*   **`HexagonLayer`**: 六角形のヒートマップを描画します。
*   **`GridLayer`**: グリッドベースのヒートマップを描画します。

これらのレイヤーを組み合わせて、複雑な可視化を作成できます


In [150]:
import pydeck as pdk
import pandas as pd

# 表示する点のデータ
data = pd.DataFrame({
    "lon": [139.6917, 135.5023, 135.7681],  # 経度
    "lat": [35.6895, 34.6937, 35.0116],   # 緯度
})

# ScatterplotLayerの設定
scatterplot_layer = pdk.Layer(
    "ScatterplotLayer",  #"HexagonLayer"
    data=data,
    get_position=["lon", "lat"],  # 点の位置
    get_radius=10000,             # 点の半径
    get_color=[255, 0, 0],    # 点の色（赤）
    pickable=True,              # クリックを有効にする
)

# 地図の初期表示設定
initial_view_state = pdk.ViewState(
    latitude=35.0,
    longitude=137.0,
    zoom=6,
    pitch=100,
)

# PyDeckのマップを作成
map = pdk.Deck(
    layers=[scatterplot_layer],
    initial_view_state=initial_view_state,
    tooltip={"html": "<b>Location:</b> ({position[0]}, {position[1]})"}  # マウスオーバー時のツールチップ
)

# HTMLファイルとして保存
map.to_html("pydeck_simple_map.html")
print("pydeck_simple_map.html が作成されました")
map
#data

pydeck_simple_map.html が作成されました


{
  "initialViewState": {
    "latitude": 35.0,
    "longitude": 137.0,
    "pitch": 100,
    "zoom": 6
  },
  "layers": [
    {
      "@@type": "ScatterplotLayer",
      "data": [
        {
          "lat": 35.6895,
          "lon": 139.6917
        },
        {
          "lat": 34.6937,
          "lon": 135.5023
        },
        {
          "lat": 35.0116,
          "lon": 135.7681
        }
      ],
      "getColor": [
        255,
        0,
        0
      ],
      "getPosition": "@@=[lon, lat]",
      "getRadius": 10000,
      "id": "0ce95685-7a17-4262-99fb-d444239f2608",
      "pickable": true
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

### 線を描画する場合

**コードの説明**

1.  **データの準備:**
    *   `num_arcs` で、描画する弧の数を指定します。
    *   ループで、各弧の始点と終点の経度、緯度をランダムに生成します。
    *   各弧のデータを辞書形式でリスト `arc_data` に追加します。
    *   `pandas` の `DataFrame` に変換します。
2.  **`ArcLayer` の設定:**
    *   `pdk.Layer` を使用して、`ArcLayer` を作成します。
    *   `"ArcLayer"` でレイヤーの種類を指定します。
    *   `data`: 描画するデータを指定します。
    *   `get_source_position`: 弧の始点の座標を、データフレームのカラム名で指定します。
    *   `get_target_position`: 弧の終点の座標を、データフレームのカラム名で指定します。
    *   `get_source_color`: 弧の始点の色を、データフレームのカラム名で指定します。
    *   `get_target_color`: 弧の終点の色を、データフレームのカラム名で指定します。
    *   `get_width`: 弧の幅を指定します。
    *   `pickable`: マウスオーバーやクリックなどのインタラクションを有効にします。
3.  **初期表示設定:**
    *   地図の初期表示位置とズームレベルを指定します。
4.  **`Deck` オブジェクトの作成:**
    *   `pdk.Deck` で、地図全体のオブジェクトを作成します。
    *   `layers`: 描画するレイヤーをリストで指定します。
    *   `initial_view_state`: 初期表示設定を指定します。
    *   `tooltip`: マウスオーバー時に表示する情報を指定します。
5.  **HTMLファイルへの保存:**
    *   作成した地図を `to_html` メソッドでHTMLファイルとして保存します。


In [152]:
import pydeck as pdk
import pandas as pd
import numpy as np

# ランダムな弧のデータを生成
num_arcs = 10
np.random.seed(0) # 結果を固定
arc_data = []
for _ in range(num_arcs):
    start_lon = np.random.uniform(139.0, 140.0)
    start_lat = np.random.uniform(35.0, 36.0)
    end_lon = np.random.uniform(139.0, 140.0)
    end_lat = np.random.uniform(35.0, 36.0)
    arc_data.append({
        "start": [start_lon, start_lat],
        "end": [end_lon, end_lat],
        "color": [np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255)] # ランダムな色
    })

# 弧のデータをDataFrameに変換
df_arcs = pd.DataFrame(arc_data)

# ArcLayerの設定
arc_layer = pdk.Layer(
    "ArcLayer", #"LineLayer"
    data=df_arcs,
    get_source_position="start",    # 始点の座標
    get_target_position="end",      # 終点の座標
    get_source_color="color", # 始点の色の指定（ここでは全て同じにしています）
    get_target_color="color", # 終点の色の指定(ここでは全て同じにしています)
    get_color = "color",
    get_width=3,        # 弧の幅
    pickable=False
)

# 地図の初期表示設定
initial_view_state = pdk.ViewState(
    latitude=35.5,
    longitude=139.5,
    zoom=9,
    pitch=0
)

# PyDeckのマップを作成
map = pdk.Deck(
    layers=[arc_layer],
    initial_view_state=initial_view_state,
    tooltip={"html": "<b>Arc</b>"}
)

# HTMLファイルとして保存
map.to_html("pydeck_arc_map.html")
print("pydeck_arc_map.html が作成されました")
map

pydeck_arc_map.html が作成されました


{
  "initialViewState": {
    "latitude": 35.5,
    "longitude": 139.5,
    "pitch": 0,
    "zoom": 9
  },
  "layers": [
    {
      "@@type": "ArcLayer",
      "data": [
        {
          "color": [
            9,
            211,
            21
          ],
          "end": [
            139.60276337607164,
            35.5448831829969
          ],
          "start": [
            139.54881350392733,
            35.71518936637242
          ]
        },
        {
          "color": [
            193,
            230,
            39
          ],
          "end": [
            139.05671297731743,
            35.272656294580116
          ],
          "start": [
            139.3843817072927,
            35.297534606544474
          ]
        },
        {
          "color": [
            9,
            148,
            115
          ],
          "end": [
            139.92559663829266,
            35.07103605819789
          ],
          "start": [
            139.5288949197529,
            35.568044561093934
          ]
        },
        {
          "color": [
            99,
            216,
            177
          ],
          "end": [
            139.14035078041263,
            35.87008725835844
          ],
          "start": [
            139.36824153984054,
            35.95715515895304
          ]
        },
        {
          "color": [
            9,
            185,
            127
          ],
          "end": [
            139.11827442586895,
            35.639921021327524
          ],
          "start": [
            139.46147936225293,
            35.78052917628646
          ]
        },
        {
          "color": [
            114,
            183,
            28
          ],
          "end": [
            139.47360041934667,
            35.18633234332676
          ],
          "start": [
            139.75861562432235,
            35.105907607187795
          ]
        },
        {
          "color": [
            244,
            17,
            79
          ],
          "end": [
            139.61763549707587,
            35.61209572272242
          ],
          "start": [
            139.56843394886866,
            35.01878980043635
          ]
        },
        {
          "color": [
            231,
            169,
            57
          ],
          "end": [
            139.61306345788412,
            35.90234858317398
          ],
          "start": [
            139.9025984755294,
            35.449949989911225
          ]
        },
        {
          "color": [
            142,
            99,
            53
          ],
          "end": [
            139.21038256107383,
            35.12892629765485
          ],
          "start": [
            139.66676671544568,
            35.67063786961816
          ]
        },
        {
          "color": [
            47,
            127,
            244
          ],
          "end": [
            139.63427405795733,
            35.95894926862452
          ],
          "start": [
            139.32504722900836,
            35.038425426472735
          ]
        }
      ],
      "getColor": "@@=color",
      "getSourceColor": "@@=color",
      "getSourcePosition": "@@=start",
      "getTargetColor": "@@=color",
      "getTargetPosition": "@@=end",
      "getWidth": 3,
      "id": "eaf588cf-821c-483e-ac45-462bee073100",
      "pickable": false
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

In [156]:
import pydeck as pdk
import pandas as pd

def plot_nodes_pydeck(nodes,             # Dictionary of node objects where each node contains attributes like latitude and longitude
                      icon="circle",      # Icon symbol to use for the markers on the map（未使用ですが、引数として残しておきます）
                      color="#0000FF",    # Color of the marker icon （HTMLカラーコード）
                      background_color="#FFFFFF", # Background color of the marker icon （未使用）
                      ):
    # ノードデータをリストに変換
    node_list = []
    for node in nodes.values():
        node_list.append({
            "lon": node.lon,
            "lat": node.lat,
            "ID": node.ID,
            "name": node.name,
            "color": color,
            "radius": 1000
        })

    # DataFrameに変換
    df_nodes = pd.DataFrame(node_list)

    # ScatterplotLayerの設定
    scatterplot_layer = pdk.Layer(
        "ScatterplotLayer",  #"HexagonLayer"
        data= df_nodes,
        get_position=["lon", "lat"],  # 点の位置
        get_radius=10000,             # 点の半径
        get_color=[255, 0, 0],    # 点の色（赤）
        pickable=True,              # クリックを有効にする
    )
    
    # 地図の初期表示設定
    initial_view_state = pdk.ViewState(
        latitude= node_list[0]["lat"],
        longitude=node_list[0]["lon"],
        zoom=6,
        pitch=0,
    )
    
    # PyDeckのマップを作成
    map = pdk.Deck(
        layers=[scatterplot_layer],
        initial_view_state=initial_view_state,
        tooltip={"html": "<b>Location:</b> ({position[0]}, {position[1]})"}  # マウスオーバー時のツールチップ
    )
    
    return map

plot_nodes_pydeck(nodes=customers, icon='industry', color='orange', background_color='yellow')


{
  "initialViewState": {
    "latitude": 43.206898,
    "longitude": -71.537994,
    "pitch": 0,
    "zoom": 6
  },
  "layers": [
    {
      "@@type": "ScatterplotLayer",
      "data": [
        {
          "ID": "C01",
          "color": "orange",
          "lat": 43.206898,
          "lon": -71.537994,
          "name": "Concord",
          "radius": 1000
        },
        {
          "ID": "C02",
          "color": "orange",
          "lat": 40.220596,
          "lon": -74.769913,
          "name": "Trenton",
          "radius": 1000
        },
        {
          "ID": "C03",
          "color": "orange",
          "lat": 30.27467,
          "lon": -97.740349,
          "name": "Austin",
          "radius": 1000
        },
        {
          "ID": "C04",
          "color": "orange",
          "lat": 41.830914,
          "lon": -71.414963,
          "name": "Providence",
          "radius": 1000
        },
        {
          "ID": "C05",
          "color": "orange",
          "lat": 44.262436,
          "lon": -72.580536,
          "name": "Montpelier",
          "radius": 1000
        },
        {
          "ID": "C06",
          "color": "orange",
          "lat": 39.798363,
          "lon": -89.654961,
          "name": "Springfield",
          "radius": 1000
        },
        {
          "ID": "C07",
          "color": "orange",
          "lat": 41.591087,
          "lon": -93.603729,
          "name": "Des Moines",
          "radius": 1000
        },
        {
          "ID": "C08",
          "color": "orange",
          "lat": 39.157307,
          "lon": -75.519722,
          "name": "Dover",
          "radius": 1000
        },
        {
          "ID": "C09",
          "color": "orange",
          "lat": 34.746613,
          "lon": -92.288986,
          "name": "Little Rock",
          "radius": 1000
        },
        {
          "ID": "C10",
          "color": "orange",
          "lat": 35.492207,
          "lon": -97.503342,
          "name": "Oklahoma City",
          "radius": 1000
        },
        {
          "ID": "C11",
          "color": "orange",
          "lat": 32.303848,
          "lon": -90.182106,
          "name": "Jackson",
          "radius": 1000
        },
        {
          "ID": "C12",
          "color": "orange",
          "lat": 33.448143,
          "lon": -112.096962,
          "name": "Phoenix",
          "radius": 1000
        },
        {
          "ID": "C13",
          "color": "orange",
          "lat": 41.140259,
          "lon": -104.820236,
          "name": "Cheyenne",
          "radius": 1000
        },
        {
          "ID": "C14",
          "color": "orange",
          "lat": 39.048191,
          "lon": -95.677956,
          "name": "Topeka",
          "radius": 1000
        },
        {
          "ID": "C15",
          "color": "orange",
          "lat": 39.768623,
          "lon": -86.162643,
          "name": "Indianapolis",
          "radius": 1000
        },
        {
          "ID": "C16",
          "color": "orange",
          "lat": 36.16581,
          "lon": -86.784241,
          "name": "Nashville",
          "radius": 1000
        },
        {
          "ID": "C17",
          "color": "orange",
          "lat": 39.961346,
          "lon": -82.999069,
          "name": "Columbus",
          "radius": 1000
        },
        {
          "ID": "C18",
          "color": "orange",
          "lat": 40.264378,
          "lon": -76.883598,
          "name": "Harrisburg",
          "radius": 1000
        },
        {
          "ID": "C19",
          "color": "orange",
          "lat": 39.739227,
          "lon": -104.984856,
          "name": "Denver",
          "radius": 1000
        },
        {
          "ID": "C20",
          "color": "orange",
          "lat": 46.82085,
          "lon": -100.783318,
          "name": "Bismarck",
          "radius": 1000
        }
      ],
      "getColor": [
        255,
        0,
        0
      ],
      "getPosition": "@@=[lo

### **Arcs**

#### Arc distances

In [53]:
# Creating a dictionary containing distances between customers and suppliers
distances = dict()
for s, supplier in suppliers.items():                                                                   # Iterate over suppliers
    for c, customer in customers.items():                                                               # Iterate over customers
        distances[s, c] = compute_geodesic_distance(origin=customer, destination=supplier, unit='km')   # Calculate and store distance in kilometers


#### Arc costs

In [54]:
cost_unit_km = 1.2  # Cost per unit per kilometer

# Creating a dictionary containing unit costs between customers and suppliers
unit_cost = dict()
for s, supplier in suppliers.items():                                                           # Iterate over suppliers
    for c, customer in customers.items():                                                       # Iterate over customers
        unit_cost[s, c] = distances[s, c] * cost_unit_km                                        # Calculate unit cost as distance multiplied by cost per km


## **Optimization model**

### **Creating and solving the optimization model**

In [55]:
# Create a new Gurobi model
model = grb.Model("Transportation Problem")

# Creating decision variables

# Decision variable representing the flows between suppliers and customers
flow_vars = dict()
for s, supplier in suppliers.items():
    for c, customer in customers.items():
        flow_vars[s, c] = model.addVar(vtype=grb.GRB.CONTINUOUS,
                                       name="flow_vars_{0}_{1}".format(s, c))

# Creating the objective function expression
total_cost = grb.quicksum(unit_cost[s, c] * flow_vars[s, c]
                         for s, supplier in suppliers.items()
                         for c, customer in customers.items())

# Setting the objective function
model.setObjective(total_cost, grb.GRB.MINIMIZE)

# Adding demand constraints
for c, customer in customers.items():
    model.addConstr(grb.quicksum(flow_vars[s, c] for s, supplier in suppliers.items()) == customer.demand)

# Adding supply constraints
for s, supplier in suppliers.items():
    model.addConstr(grb.quicksum(flow_vars[s, c] for c, customer in customers.items()) <= supplier.supply)

# Solving the model
model.optimize()


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/mikiokubo/Documents/dev/mit-scd/.venv/lib/python3.10/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/ebc7126a9a784f259bd2c8a1349336ab-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/ebc7126a9a784f259bd2c8a1349336ab-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 31 COLUMNS
At line 392 RHS
At line 419 BOUNDS
At line 540 ENDATA
Problem MODEL has 26 rows, 120 columns and 240 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 26 (0) rows, 120 (0) columns and 240 (0) elements
0  Obj 0 Primal inf 171000 (20)
30  Obj 1.4374377e+08
Optimal - objective value 1.4374377e+08
Optimal objective 143743768.1 - 30 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU seconds):

/Users/mikiokubo/Documents/dev/mit-scd/.venv/lib/python3.10/site-packages/pulp/pulp.py:1298: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


### **Alternative formulation for the optimization model**

In [56]:
# Create a new Gurobi model
model = grb.Model("Transportation Problem")

# Creating decision variables

# Decision variable representing the flows between suppliers and customers
flow_vars = {(s, c): model.addVar(vtype=grb.GRB.CONTINUOUS,
                                  name="flow_vars_{0}_{1}".format(s, c))
             for s in suppliers.keys()
             for c in customers.keys()
             }

# Creating the objective function expression
total_cost = grb.quicksum(flow_vars[s, c] * unit_cost[s, c]
                         for s in suppliers.keys() for c in customers.keys())

# Setting the objective function
model.setObjective(total_cost, grb.GRB.MINIMIZE)

# Adding demand constraints
demand_constraints = {c: model.addConstr(grb.quicksum(flow_vars[s, c] for s in suppliers.keys()) == customers[c].demand)
                      for c in customers.keys()
                      }

# Adding supply constraints
supply_constraints = {s: model.addConstr(grb.quicksum(flow_vars[s, c] for c in customers.keys()) <= suppliers[s].supply)
                      for s in suppliers.keys()
                      }

# Solving the model
model.optimize()


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/mikiokubo/Documents/dev/mit-scd/.venv/lib/python3.10/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/9c19502a65c94026b28ff836d84a5e2c-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/9c19502a65c94026b28ff836d84a5e2c-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 31 COLUMNS
At line 392 RHS
At line 419 BOUNDS
At line 540 ENDATA
Problem MODEL has 26 rows, 120 columns and 240 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 26 (0) rows, 120 (0) columns and 240 (0) elements
0  Obj 0 Primal inf 171000 (20)
30  Obj 1.4374377e+08
Optimal - objective value 1.4374377e+08
Optimal objective 143743768.1 - 30 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU seconds):

## **Solution visualization and analysis**

In [57]:
# Visualizing the customer and supplier locations on the map

# Create a new map centered on the US with a zoom level of 5
map = folium.Map([40, -95.0], zoom_start=5)

# Plot customer locations with a store icon, green color, and yellow background
plot_nodes(map=map, nodes=customers, icon='store', background_color='yellow', color='green')

# Plot supplier locations with an industry icon, orange color, and yellow background
plot_nodes(map=map, nodes=suppliers, icon='industry', background_color='yellow', color='orange')

# Plot the flows between suppliers and customers with a maximum line width of 20
plot_flows(map=map, max_width=20, vars=flow_vars, nodes=nodes)

# Add a tile layer for better map visualization (cartodbpositron theme)
folium.TileLayer('cartodbpositron').add_to(map)

# Display the map with all the plotted data
map


In [91]:
import pydeck as pdk
import numpy as np
import pandas as pd

# ランダムな点のデータ生成 (例として10万点)
num_points = 10000
np.random.seed(0)  # 結果を固定するため

# RGBカラーの配列を生成（各要素は[R, G, B]のリスト）
colors = np.random.randint(0, 255, size=(num_points, 3)).tolist()


data = pd.DataFrame({
    "lon": np.random.uniform(139.0, 140.0, num_points),
    "lat": np.random.uniform(35.0, 36.0, num_points),
    "radius": np.random.uniform(100, 200, num_points), # 点の半径
    "color": colors, # RGBカラーをリストとして格納
})

# PyDeckのScatterplotLayerを設定
scatterplot_layer = pdk.Layer(
    "ScatterplotLayer",
    data=data,
    get_position=["lon", "lat"],
    get_radius="radius",
    get_color="color",
    pickable=True,
    opacity=0.8
)

# 地図の初期表示設定
initial_view_state = pdk.ViewState(
    latitude=35.5,
    longitude=139.5,
    zoom=10,
    pitch=0
)

# PyDeckのマップを作成
map = pdk.Deck(
    layers=[scatterplot_layer],
    initial_view_state=initial_view_state,
    tooltip={"html": "<b>Location:</b> ({position[0]}, {position[1]})"}
)

# HTMLファイルとして保存
map.to_html("pydeck_map.html")


In [87]:
import pandas as pd
import pydeck as pdk

DATA_URL = "https://raw.githubusercontent.com/visgl/deck.gl-data/master/website/bart-lines.json"
df = pd.read_json(DATA_URL)


def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i : i + 2], 16) for i in (0, 2, 4))


df["color"] = df["color"].apply(hex_to_rgb)


view_state = pdk.ViewState(latitude=37.782556, longitude=-122.3484867, zoom=10)

layer = pdk.Layer(
    type="PathLayer",
    data=df,
    pickable=True,
    get_color="color",
    width_scale=20,
    width_min_pixels=2,
    get_path="path",
    get_width=5,
)

r = pdk.Deck(layers=[layer], initial_view_state=view_state, tooltip={"text": "{name}"})
r.to_html("path_layer.html")